# 02 — BM25 baseline

Build the first measurable retrieval system. BM25 ranks passages using word frequency, word rarity, and passage length. It needs no embedding model or external API.

**Pipeline:** original contract text → 1,000-character chunks with 200-character overlap → BM25 → ranked passages with document paths and offsets.

These are initial, untuned settings. We search every selected contract; gold document IDs and answer spans are used only after retrieval for scoring. This notebook reads train questions only.

In [1]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import platform
import sys
from time import perf_counter

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from retrieval import BM25Retriever, evaluate_train, load_inputs

CONFIG_PATH = ROOT / 'configs' / 'bm25_baseline.json'
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
display(pd.Series(config, name='baseline settings'))

name             bm25_baseline
chunk_size                1000
chunk_overlap              200
k1                         1.5
b                         0.75
min_score                  0.0
k_values         [1, 3, 5, 10]
Name: baseline settings, dtype: object

## 1. Load verified data and build the index

Only tokenization lowercases text. Stored passages remain exact slices of the original document. Hash checks detect changes to the prepared train data or corpus.

Change settings through configuration when experimenting later. Preserve this baseline configuration and give each new approach a separate name.

In [2]:
train, documents, manifest = load_inputs(ROOT)
started = perf_counter()
retriever = BM25Retriever(documents, config)
build_seconds = perf_counter() - started
assert all(c['text'] == documents[c['file_path']][c['start']:c['end']] for c in retriever.chunks)
print(f'{len(train)} train questions | {len(documents)} contracts | {len(retriever.chunks)} chunks')
print(f'Index build: {build_seconds:.3f} seconds')

140 train questions | 29 contracts | 1522 chunks
Index build: 0.153 seconds


## 2. Try a question

Edit `question` and rerun this cell during a walkthrough. Each result includes a BM25 score, source, offsets, and exact passage text. Scores rank passages; they are not probabilities.

The baseline returns no results if no score exceeds zero. This basic rule is not calibrated for realistic unanswerable questions.

In [3]:
question = train[0]['query']
hits = retriever.search(question, k=3)
print(question)
if hits:
    display(pd.DataFrame(hits)[['score', 'file_path', 'start', 'end', 'text']])
else:
    print('No relevant passage found.')

Consider the Amendment No. 3 to Wireless Content License Agreement between Fox Mobile Entertainment, Inc. and Glu Mobile Inc.; What is the expiration date of this contract?


,score,file_path,start,end,text
0,81.547916,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,0,1000,"AMENDMENT NO. 3\n\nDated as of February 19, 20..."
1,61.647242,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,800,1800,"dom of Heaven, Mr. and Mrs. Smith, In Her Shoe..."
2,59.824400,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,1600,1902,"x Mobile Entertainment, Inc. (""Fox"")\n\n\n\nBy..."


## 3. Measure train retrieval

For each question, treat characters as `(document, position)` pairs. Merge overlapping retrieved intervals and overlapping gold intervals before computing:

- **Character precision:** relevant retrieved characters / all retrieved characters.
- **Character recall:** relevant retrieved characters / all gold characters.
- **Hit rate:** fraction of questions with any gold-character overlap (not complete answers).

Report the mean per-query score at each k; an empty result scores zero. Increasing k can improve coverage while making reviewers read more irrelevant text. See the [LegalBench-RAG paper](https://arxiv.org/abs/2408.10343) for benchmark background.

Latency covers query tokenization, scoring, ranking, and result creation on this machine. It excludes index building and evaluation; these are single-run measurements, not a production load test.

In [4]:
result = evaluate_train(retriever, train, config['k_values'])
metrics = pd.DataFrame(result['metrics'])
display(metrics.style.format({'precision': '{:.2%}', 'recall': '{:.2%}', 'hit': '{:.2%}'}))
print('Search latency (ms):', result['latency_ms'])
print('False abstention on answerable train questions:', result['answerable_false_abstention_rate'])
print('Correct abstention on unanswerable questions: not measured; validated set pending.')

,k,n_queries,precision,recall,hit
0,1,140,0.56%,1.72%,3.57%
1,3,140,1.36%,7.43%,12.14%
2,5,140,1.74%,18.16%,26.43%
3,10,140,1.81%,32.16%,43.57%


Search latency (ms): {'median': 9.281850012484938, 'p95': 13.42193005257286}
False abstention on answerable train questions: 0.0
Correct abstention on unanswerable questions: not measured; validated set pending.


## 4. Inspect train failures

Start with questions having the least recall at k=5. Compare predicted passages with gold only after retrieval. These diagnostics can guide later experiments; this notebook does not tune any settings.

In [5]:
per_query = pd.DataFrame(result['per_query'])
worst = per_query[per_query['k'] == 5].sort_values(['recall', 'precision']).head(10)
questions = pd.DataFrame(train)[['query_id', 'query']]
display(worst.merge(questions, on='query_id'))
example_id = worst.iloc[0]['query_id']
example = next(r for r in train if r['query_id'] == example_id)
prediction = next(p for p in result['predictions'] if p['query_id'] == example_id)
print('QUESTION:', example['query'])
print('GOLD:')
display(pd.DataFrame(example['snippets']))
print('TOP PREDICTIONS:')
display(pd.DataFrame(prediction['hits'][:3]))

,query_id,k,precision,recall,hit,query
0,b5a0948cd3cf1a4070748a4899cab8bea0dde912b91f0f...,5,0.0,0.0,0,Consider the Service Agreement between Bicycle...
1,5aacd88644962ff51293f4cc0ae5f18f1815d9a615908b...,5,0.0,0.0,0,Consider the Service Agreement between Bicycle...
2,13d8e76465cd5d5ff8d880d4fcf36657d918c9f750e2aa...,5,0.0,0.0,0,Consider the Service Agreement between Bicycle...
3,505f5239e0ca460f383a732198b07e7d4dd7f828487ada...,5,0.0,0.0,0,Consider the Service Agreement between Bicycle...
4,41ece44a0165f292b5f7c0cd8f4402ff9844983a576937...,5,0.0,0.0,0,Consider the Cooperation Agreement between HPI...
5,87057f5be40e9ab12f44828a63459c2bb49f49f5f6fe68...,5,0.0,0.0,0,Consider the Cooperation Agreement between HPI...
6,600ab72c4d32f01160c1ca482e07a72d3ee3138a0f9642...,5,0.0,0.0,0,Consider the Cooperation Agreement between HPI...
7,70069a2599a5b9df39ec8e006a110e01d374cea11133eb...,5,0.0,0.0,0,Consider the Sponsorship Agreement between Jac...
8,83b1521f8cc3a39a206573315d1427eeb6dc73dfd5665f...,5,0.0,0.0,0,Consider the Service Agreement between Therava...
9,434f75622161ca957009e403b6bf7a4c845418d14c3205...,5,0.0,0.0,0,Consider the Service Agreement between Therava...


QUESTION: Consider the Service Agreement between BicycleTX Ltd and Nigel Crockett for Chief Business Officer Position; What is the governing law for this contract?
GOLD:


,file_path,span,answer
0,cuad/BICYCLETHERAPEUTICSPLC_03_10_2020-EX-10.1...,"[53956, 54252]",This Agreement shall be governed by and interp...


TOP PREDICTIONS:


,file_path,start,end,text,score
0,cuad/BICYCLETHERAPEUTICSPLC_03_10_2020-EX-10.1...,0,1000,Exhibit 10.11\n\nDATED 26 September 2019\n\nBi...,49.792763
1,cuad/BICYCLETHERAPEUTICSPLC_03_10_2020-EX-10.1...,56000,57000,"atisfied, the Acquiring Person will have Contr...",38.146867
2,cuad/HealthcareIntegratedTechnologiesInc_20190...,20800,21800,"n, or shipment of the Products. Neither party ...",30.389502


## 5. Save a baseline run

Each execution creates a new run directory, preserving previous measurements. Saved predictions include passage text and offsets so results can be audited. No test evaluation is performed.

The manifest identifies data and code used for this run. Timing varies across runs. The unanswerable metric is explicitly null until that dataset is validated. This is a retrieval baseline checkpoint, not yet the complete core application.

In [6]:
from datetime import datetime, timezone

run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
output = ROOT / 'results' / config['name'] / run_id
output.mkdir(parents=True, exist_ok=False)
metadata = {
    'run_id': run_id, 'split': 'train', 'test_evaluation_runs': 0,
    'dataset_revision': manifest['revision'], 'config': config,
    'build_seconds': build_seconds, 'chunk_count': len(retriever.chunks),
    'prepared_manifest_sha256': hashlib.sha256((ROOT / 'data/prepared/manifest.json').read_bytes()).hexdigest(),
    'code_sha256': hashlib.sha256((ROOT / 'retrieval.py').read_bytes()).hexdigest(),
    'python': platform.python_version(), 'platform': platform.platform(),
    'packages': {p: importlib.metadata.version(p) for p in ['rank-bm25', 'numpy', 'pandas']},
}
for name, value in [('run', metadata), ('metrics', {k: v for k, v in result.items() if k not in ['predictions', 'per_query']}),
                    ('predictions', result['predictions'])]:
    (output / f'{name}.json').write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')
metrics.to_csv(output / 'metrics.csv', index=False)
per_query.to_csv(output / 'per_query.csv', index=False)
print('Saved run:', output)

Saved run: C:\Users\vatsa\OneDrive\Desktop\JioStar\results\bm25_baseline\20261101T020109139330Z


## Explain this step

We built a local keyword retrieval baseline with traceable sources. Character recall measures coverage of the required clauses; precision measures how much retrieved text is relevant. The baseline provides evidence for deciding whether later complexity is worthwhile.

**Current limits:** fixed chunks can split clauses; lexical matching can miss paraphrases; generic terms may retrieve the wrong section. Zero-score abstention does not establish answerability. No web app, calibrated abstention, improvement loop, or LLM has been added yet.

**Next checkpoint:** create and validate at least 30 unanswerable questions, freeze and commit them before improvements, and declare the improvement objective and guardrails. Keep test performance untouched.